# 34 - Qwen3-32B Concise Evaluation Prompt Ablation

This notebook keeps the final retrieval stack fixed and tests only a shorter, benchmark-style answer prompt for lexical metrics such as Token F1 and ROUGE-L.

It uses the same locked benchmark and the same precomputed final retrieval output used by notebook 13.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys
import json
import pandas as pd
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
os.chdir(DRIVE_ROOT)
if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Working directory:', Path.cwd())
print('Device:', device)

In [ ]:
# Install only if the Colab runtime is missing packages.
try:
    import transformers, peft, bitsandbytes, faiss, rank_bm25
    print('Core dependencies already available.')
except Exception:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

In [ ]:
from src.generation import load_llm, generate_text, load_precomputed_retrieval
from src.prompting import build_concise_eval_rag_prompt
from src.retrieval import RetrievalEngine
from src.evaluation_qa import evaluate_generation_predictions

benchmark_csv = DRIVE_ROOT / 'data/benchmark/gold_benchmark_v1.csv'
index_root = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b'
precomputed_retrieval_csv = DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_dense_top30_qwen3_reranker_8b_predictions_v1.csv'
llm_model = 'Qwen/Qwen3-32B'

output_predictions_csv = DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_concise_prompt_predictions_v1.csv'
output_eval_csv = DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_concise_prompt_eval_v1.csv'
output_summary_json = DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_concise_prompt_summary_v1.json'
output_run_config_json = DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_concise_prompt_run_config_v1.json'

for path in [benchmark_csv, index_root / 'index_manifest.json', precomputed_retrieval_csv]:
    if not path.exists():
        raise FileNotFoundError(path)

benchmark_csv, index_root, precomputed_retrieval_csv

In [ ]:
# Optional smoke test. Set to None for the full 190-question run.
LIMIT = 10
MAX_NEW_TOKENS = 256
MAX_CONTEXT_CHARS = 14000
INPUT_MAX_LENGTH = 8192

def run_concise_prompt_generation(limit=None):
    benchmark = pd.read_csv(benchmark_csv, dtype=str, keep_default_na=False)
    if limit:
        benchmark = benchmark.head(limit).copy()

    engine = RetrievalEngine(index_root=index_root, device=device)
    precomputed = load_precomputed_retrieval(precomputed_retrieval_csv, engine.metadata)

    tokenizer, model = load_llm(llm_model, device=device, load_in_4bit=True)
    rows = []
    try:
        from tqdm.auto import tqdm
        for _, item in tqdm(benchmark.iterrows(), total=len(benchmark), desc='Concise prompt generation'):
            question = item['question']
            question_id = str(item.get('question_id', '')).strip()
            retrieved = precomputed.get(question_id, [])[:10]
            prompt = build_concise_eval_rag_prompt(
                question,
                retrieved,
                max_context_chars=MAX_CONTEXT_CHARS,
            )
            answer = generate_text(
                tokenizer=tokenizer,
                model=model,
                prompt=prompt,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.0,
                top_p=1.0,
                input_max_length=INPUT_MAX_LENGTH,
            )
            rows.append({
                'question_id': item.get('question_id', ''),
                'topic': item.get('topic', ''),
                'difficulty': item.get('difficulty', ''),
                'question': question,
                'gold_answer': item.get('gold_answer', ''),
                'gold_doc_keys': item.get('gold_doc_keys', ''),
                'gold_article_keys': item.get('gold_article_keys', ''),
                'gold_law': item.get('gold_law', ''),
                'gold_article_no': item.get('gold_article_no', ''),
                'generated_answer': answer,
                'retrieved_article_keys': '; '.join(str(row.get('article_key', '')) for row in retrieved),
                'retrieved_doc_keys': '; '.join(str(row.get('doc_key', '')) for row in retrieved),
                'retrieved_citations': ' | '.join(str(row.get('citation_label', '')) for row in retrieved),
                'prompt_version': 'concise_eval_tr_v1',
                'llm_model': llm_model,
                'adapter_path': '',
                'system_name': 'qwen3_32b_base_rag_concise_prompt',
                'retriever_mode': 'dense',
                'reranker_model': 'Qwen/Qwen3-Reranker-8B',
                'precomputed_retrieval_csv': str(precomputed_retrieval_csv),
                'top_k_context': 10,
            })
    finally:
        del tokenizer
        del model
        del engine
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    output_predictions_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(output_predictions_csv, index=False, encoding='utf-8-sig')
    output_run_config_json.write_text(json.dumps({
        'benchmark_csv': str(benchmark_csv),
        'index_root': str(index_root),
        'precomputed_retrieval_csv': str(precomputed_retrieval_csv),
        'output_predictions_csv': str(output_predictions_csv),
        'llm_model': llm_model,
        'prompt_version': 'concise_eval_tr_v1',
        'limit': limit,
        'max_new_tokens': MAX_NEW_TOKENS,
        'max_context_chars': MAX_CONTEXT_CHARS,
        'input_max_length': INPUT_MAX_LENGTH,
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    return output_predictions_csv

predictions_path = run_concise_prompt_generation(limit=LIMIT)
predictions_path

In [ ]:
summary = evaluate_generation_predictions(
    predictions_csv=output_predictions_csv,
    output_eval_csv=output_eval_csv,
    output_summary_json=output_summary_json,
)
print(json.dumps(summary, ensure_ascii=False, indent=2))

pd.read_csv(output_eval_csv).head(10)

In [ ]:
retrieval_summary_json = DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_dense_top30_qwen3_reranker_8b_summary_v1.json'

rows = []
if retrieval_summary_json.exists():
    retrieval_summary = json.loads(retrieval_summary_json.read_text(encoding='utf-8'))
    retrieval_metrics = retrieval_summary.get('metrics', {})
    for metric in [
        'doc_hit@5', 'doc_hit@10', 'article_hit@5', 'article_hit@10',
        'doc_recall@5', 'doc_recall@10', 'article_recall@5', 'article_recall@10',
        'doc_mrr', 'article_mrr', 'doc_ndcg@5', 'doc_ndcg@10',
        'article_ndcg@5', 'article_ndcg@10',
    ]:
        if metric in retrieval_metrics:
            rows.append({'section': 'retrieval_fixed_from_notebook_12', 'metric': metric, 'value': retrieval_metrics[metric]})
else:
    print('Retrieval summary not found:', retrieval_summary_json)

if output_summary_json.exists():
    generation_summary = json.loads(output_summary_json.read_text(encoding='utf-8'))
    generation_metrics = generation_summary.get('metrics', {})
    for metric in [
        'exact_match', 'token_f1', 'rouge_l', 'retrieval_gold_available',
        'citation_present', 'citation_gold_match', 'grounded_citation_score',
        'unsupported_or_missing_citation',
    ]:
        if metric in generation_metrics:
            rows.append({'section': 'generation_concise_prompt', 'metric': metric, 'value': generation_metrics[metric]})
else:
    print('Generation summary not found:', output_summary_json)

combined_metrics = pd.DataFrame(rows)
combined_output_csv = DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_concise_prompt_combined_metrics_v1.csv'
combined_output_csv.parent.mkdir(parents=True, exist_ok=True)
combined_metrics.to_csv(combined_output_csv, index=False, encoding='utf-8-sig')
print('Saved combined metrics:', combined_output_csv)
combined_metrics